# STEP 1: Overview of Script
# ---------------------------------------
# This script automates the assignment of roof constructions in ESP-r models.
#
# Workflow:
# 1. Load EPC dataset (contains BUILDING_REFERENCE_NUMBER and WALL_CONS).
# 2. Load mapping file (WALL_CONS → ESP-r assignment characters).
#    - Handles both single-letter and multi-line assignments (e.g. "0\nz").
# 3. Loop through each building folder in baseline_models.
# 4. Identify the model subfolder (e.g. Detached_2F, Flat, etc.).
# 5. Locate the cfg/roof.txt file.
# 6. Replace all instances of "WALL" in walls.txt with the correct assignment.
# 7. Save the updated file.
#
# Notes:
# - Buildings without walls.txt are skipped.
# - Multi-line assignments preserve correct line breaks (critical for ESP-r).
# - Designed for use in Jupyter Notebook on macOS.

In [ ]:
# STEP 2: Import Libraries and Define Paths
# ---------------------------------------
import os
import pandas as pd

# File paths
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
mapping_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/wall_ESP-r_assign.csv"
baseline_dir = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

In [2]:
# STEP 3: Load EPC Dataset
# ---------------------------------------
epc_df = pd.read_csv(epc_path)

# Ensure correct columns exist
required_cols = ["BUILDING_REFERENCE_NUMBER", "WALL_CONS"]
for col in required_cols:
    if col not in epc_df.columns:
        raise ValueError(f"Missing column: {col}")

# Convert building ID to string for matching folder names
epc_df["BUILDING_REFERENCE_NUMBER"] = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str)

print(f"EPC dataset loaded: {len(epc_df)} records")

EPC dataset loaded: 117 records


In [4]:
# STEP 4 (FINAL FIX): Proper CSV Parsing with Multi-line Support
# ---------------------------------------
mapping_dict = {}

with open(mapping_path, "r") as f:
    content = f.read()

# Split into logical CSV rows (handle quoted multiline properly)
rows = []
current = ""
in_quotes = False

for char in content:
    if char == '"':
        in_quotes = not in_quotes

    if char == "\n" and not in_quotes:
        rows.append(current.strip())
        current = ""
    else:
        current += char

# Add last row
if current:
    rows.append(current.strip())

# Remove header
rows = rows[1:]

# Parse rows
for row in rows:
    if "," not in row:
        continue

    key, value = row.split(",", 1)

    key = key.strip()
    value = value.strip()

    # Remove surrounding quotes if present
    if value.startswith('"') and value.endswith('"'):
        value = value[1:-1]

    # Replace literal newlines correctly
    value = value.replace("\r", "").strip()

    mapping_dict[key] = value

# Debug check
print(f"Loaded {len(mapping_dict)} mappings:\n")
for k, v in list(mapping_dict.items())[:31]:
    print(f"{k} -> {repr(v)}")

Loaded 31 mappings:

avg_u0.12 -> 'a'
avg_u0.13 -> 'f'
avg_u0.21 -> 'g'
brick_as_built_bands_A_to_C -> 'v'
cavity_as_built_band_G -> 'i'
cavity_as_built_bands_A_to_E -> 'h'
cavity_ext_insul_band_D -> 'm'
cavity_filled_band_F -> 'k'
cavity_filled_band_G -> 'l'
cavity_filled_bands_A_to_E -> 'j'
gran_whin_as_built_band_A -> 'n'
gran_whin_int_insul_band_A -> 'p'
gran_whin_int_insul_band_G -> 'q'
park_home_as_built_band_G -> '0\ne'
park_home_as_built_band_I -> '0\nf'
park_home_ext_insul_band_F -> '0\ng'
park_home_ext_insul_band_G -> '0\nh'
park_home_ext_insul_band_I -> '0\ni'
park_home_ext_insul_band_K -> '0\nj'
sand_lime_as_built_band_A -> 'r'
sand_lime_as_built_band_C -> 'g'
sand_lime_as_built_band_G -> 't'
sand_lime_int_insul_band_J -> 'u'
system_as_built_band_D -> 'w'
system_as_built_insulated -> '0\nx'
system_ext_insul_band_D -> '0\ny'
timber_as_built_band_D -> '0\nz'
timber_as_built_band_F -> '0\na'
timber_as_built_band_J -> '0\nc'
timber_as_built_band_K -> '0\nd'
timber_as_built_band

In [4]:
#VALIDATION CHECK: Ensure all keys in mapping_dict exist in EPC dataset
print("\nTEST LOOKUPS:")
test_keys = [
    "cavity_filled_bands_A_to_E",
    "timber_as_built_bands_G_to_I"
]

for key in test_keys:
    print(f"{key} -> {repr(mapping_dict.get(key))}")


TEST LOOKUPS:
cavity_filled_bands_A_to_E -> 'j'
timber_as_built_bands_G_to_I -> '0\nb'


In [5]:
# STEP 5: Define Possible Model Folder Names
# ---------------------------------------
model_folder_names = [
    "SemiDetached_2F", "Detached_2F"
]

In [6]:
# STEP 6: Process Each Building
# ---------------------------------------
processed = 0
skipped = 0

for _, row in epc_df.iterrows():
    building_id = row["BUILDING_REFERENCE_NUMBER"]
    wall_cons = row["WALL_CONS"]

    # Get assignment from mapping
    assignment = mapping_dict.get(wall_cons)

    if assignment is None:
        print(f"Skipping {building_id}: No mapping for {wall_cons}")
        skipped += 1
        continue

    building_path = os.path.join(baseline_dir, building_id)

    if not os.path.isdir(building_path):
        print(f"Skipping {building_id}: Folder not found")
        skipped += 1
        continue

    # Find model folder
    model_folder = None
    for name in model_folder_names:
        candidate = os.path.join(building_path, name)
        if os.path.isdir(candidate):
            model_folder = candidate
            break

    if model_folder is None:
        print(f"Skipping {building_id}: No model folder found")
        skipped += 1
        continue

    # Path to walls.txt
    walls_txt_path = os.path.join(model_folder, "cfg", "walls.txt")

    if not os.path.isfile(walls_txt_path):
        print(f"Skipping {building_id}: walls.txt not found")
        skipped += 1
        continue

    # Read file
    with open(walls_txt_path, "r") as f:
        content = f.read()

    # Replace placeholder
    updated_content = content.replace("WALL", assignment)

    # Save file
    with open(walls_txt_path, "w") as f:
        f.write(updated_content)

    processed += 1

print("\n--- Processing Complete ---")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")

Skipping 1001829071: No model folder found
Skipping 1001991633: No model folder found
Skipping 1001063639: No model folder found
Skipping 1234761000: No model folder found
Skipping 1001906271: No model folder found
Skipping 1001829057: No model folder found
Skipping 1234760999: No model folder found
Skipping 1002143357: No model folder found
Skipping 1001951854: No model folder found
Skipping 1001829069: No model folder found
Skipping 1002313096: No model folder found
Skipping 1002143351: No model folder found
Skipping 1001870864: No model folder found
Skipping 1002143293: No model folder found
Skipping 1002143352: No model folder found
Skipping 1234647955: No model folder found
Skipping 1002313093: No model folder found
Skipping 1001991629: No model folder found
Skipping 1001991628: No model folder found
Skipping 1234761002: No model folder found
Skipping 1002143355: No model folder found
Skipping 1001906269: No model folder found
Skipping 1001870855: No model folder found
Skipping 10

In [7]:
# STEP 7: Optional Validation (Check a Few Files)
# ---------------------------------------
# Randomly print a few updated files to verify correctness
import random

sample_ids = random.sample(list(epc_df["BUILDING_REFERENCE_NUMBER"]), 5)

for building_id in sample_ids:
    building_path = os.path.join(baseline_dir, building_id)

    for name in model_folder_names:
        model_folder = os.path.join(building_path, name)
        walls_txt_path = os.path.join(model_folder, "cfg", "walls.txt")

        if os.path.isfile(walls_txt_path):
            print(f"\n--- {building_id} ---")
            with open(walls_txt_path, "r") as f:
                print(f.read())
            break


--- 1001063642 ---
m
c
a
a
f
*
b
a
h
a
b
c
d
-
y
b
-
!

y
-
y
b
f
*
b
a
h
a
b
c
d
-
y
b
-
!

y
-
y
c
f
*
b
a
h
a
c
-
y
b
-
!

y
-
y
-
-
!


-
-


--- 1234982990 ---
m
c
a
a
f
*
b
a
g
a
b
c
d
-
y
b
-
!

y
-
y
b
f
*
b
a
g
a
b
c
d
-
y
b
-
!

y
-
y
c
f
*
b
a
g
a
c
-
y
b
-
!

y
-
y
-
-
!


-
-


--- 1001829065 ---
m
c
a
a
f
*
b
a
j
a
b
c
d
-
y
b
-
!

y
-
y
b
f
*
b
a
j
a
b
c
d
-
y
b
-
!

y
-
y
c
f
*
b
a
j
a
c
-
y
b
-
!

y
-
y
-
-
!


-
-


--- 1001627539 ---
m
c
a
a
f
*
b
a
0
i
a
b
c
d
-
y
b
-
!

y
-
y
b
f
*
b
a
0
i
a
b
c
d
-
y
b
-
!

y
-
y
c
f
*
b
a
0
i
a
c
-
y
b
-
!

y
-
y
-
-
!


-
-

